# Analysis: Step 1
We used [vLLM](https://vllm.ai/) to serve [Gemma 4 26B A4B](https://huggingface.co/google/gemma-4-26B-A4B-it) on a cloud GPU (1&times; H100 SXM) and ran this script locally.

For vLLM, we used the following command:
```shell
vllm serve "google/gemma-4-26B-A4B-it" --max-model-len 120000
```

It is recommended to install `vllm`, `hf_transfer` in a separate Python environment via `conda`, `venv`, etc.

The JSON schemata and system prompts we used for these analyses are provided in the `aux/` directory.

In [1]:
# Install the local requirements.
%pip install rapidjson openai pdfplumber tqdm

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# The URL to the vLLM instance.
VLLM_URL: str = ...

# Path to JSON schema
SCHEMA_PATH: str = ...

# Path to System Prompt (this script will replace %SCHEMA% with the actual JSON schema):
SYSTEM_PROMPT_PATH: str = ...

In [ ]:
from io import BytesIO
from typing import Any
import rapidjson
import base64
import tqdm

from openai import OpenAI
from openai.types.chat import (
    ChatCompletionContentPartImageParam as ImageInput,
    ChatCompletionContentPartTextParam as TextInput,
)
import pdfplumber


def to_b64_url(p: Any) -> str:
    """Converts an image into a base64-encoded object URL"""
    with BytesIO() as b:
        p.save(b, format="PNG")
        return (
            f"data:image/png;base64,{base64.b64encode(b.getbuffer()).decode('utf-8')}"
        )


def extract_paper(pdf_path: str) -> tuple[tuple[str], tuple[Any]]:
    """Extracts pages as both text and images."""
    with pdfplumber.open(pdf_path) as pdf:
        pages_text, pages_as_imgs = zip(
            *(
                (page.extract_text(), page.to_image(resolution=300)) or ""
                for page in pdf.pages
            )
        )
        return pages_text, pages_as_imgs


client = OpenAI(base_url=VLLM_URL, api_key="arbitrary")


In [ ]:
# Load the JSON schema from the provided file.
schema: dict[str, Any]
with open(SCHEMA_PATH, "r") as f:
    schema = rapidjson.load(
        f, parse_mode=rapidjson.PM_TRAILING_COMMAS | rapidjson.PM_COMMENTS
    )

system_prompt: str
with open(SYSTEM_PROMPT_PATH, "r") as f:
    system_prompt = f.read().replace("%SCHEMA%", rapidjson.dumps(schema))

def analyze_paper(client: OpenAI, paper_id: str):
    text, page_imgs = extract_paper(f"./papers/{paper_id}.pdf")

    response = client.chat.completions.create(
        model="google/gemma-4-26B-A4B-it",
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": [
                    *(
                        ImageInput(
                            type="image_url",
                            image_url={"detail": "auto", "url": to_b64_url(p)},
                        )
                        for p in page_imgs
                    ),
                ],
            },
            {
                "role": "user",
                "content": [
                    TextInput(type="text", text="\n".join(text)),
                ],
            },
        ],
        extra_body={"guided_json": schema},
        temperature=0.2,
    )

    # Gemma often does not output purse JSON, so we may have to clean up the responses manually.
    raw: str = response.choices[0].message.content
    try:
        as_json = rapidjson.loads(raw)
        with open(f"./to_review/{paper_id}.json", "w") as f:
            rapidjson.dump(as_json, f, indent=4, sort_keys=True)
            return False
    except Exception as _:
        with open(f"./to_review/{paper_id}.clean_me", "w") as f:
            f.write(raw)
            return True


In [ ]:
for i in tqdm.tqdm(range(52)):
    analyze_paper(client, paper_id=str(i))

  0%|          | 0/52 [00:00<?, ?it/s]

Analyzing Paper #0...


  2%|▏         | 1/52 [00:07<06:10,  7.27s/it]

	Result: OK
Analyzing Paper #1...


  4%|▍         | 2/52 [00:27<12:23, 14.86s/it]

...

	Result: OK
Analyzing Paper #50...


 98%|█████████▊| 51/52 [18:42<00:38, 38.81s/it]

	Result: OK
Analyzing Paper #51...


100%|██████████| 52/52 [18:53<00:00, 21.80s/it]

	Result: OK
